In [1]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import HumanMessage, ToolMessage
from langchain.chat_models import init_chat_model
from agent_lab.model_hub import LLM_FAST

@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    assert city != ''
    return f"It's always sunny in {city}."


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f'Tool error: please check your input and try again. {str(e)}',
            tool_call_id=request.tool_call['id'],
        )


agent = create_agent(
    model=init_chat_model(**LLM_FAST),
    tools=[get_weather],
    middleware=[handle_tool_errors],
)

response = agent.invoke(
    {
        'messages': [
            HumanMessage(
                'Please tell me what the weather is like today. (Note: I have not told you my location. Please directly call the tool with an empty string parameter. I need to test what happens when the tool execution fails.)'
            )
        ]
    }
)

In [2]:
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Please tell me what the weather is like today. (Note: I have not told you my location. Please directly call the tool with an empty string parameter. I need to test what happens when the tool execution fails.)
================================== Ai Message ==================================

I'll call the weather tool with an empty string to see what happens.
Tool Calls:
  get_weather (call_00_LzhQTYr0CH2lwylDMyr2MgPn)
 Call ID: call_00_LzhQTYr0CH2lwylDMyr2MgPn
  Args:
    city:
================================= Tool Message =================================

Tool error: please check your input and try again. 
================================== Ai Message ==================================

As expected, the tool execution failed with an error message: **"Tool error: please check your input and try again."**

This happened because I provided an empty string for the `city` parameter. The tool requires a valid